In [1]:
# --------------------------
# 新增方法：基于 CUSUM 的电压检测方法 (Voltage CUSUM Detector)
# 该方法利用累积和检测电压均值相对于正常状态基线的偏离情况，
# 当累计偏差超过设定阈值时判定为 outage（异常）。
from sklearn.ensemble import RandomForestClassifier
import numpy as np
import matplotlib.pyplot as plt
from datasets import *
from sklearn.metrics import recall_score, f1_score, roc_auc_score, accuracy_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
import shap
import reload_shap_plot

In [2]:
class VoltageCUSUMDetector:
    def __init__(self, feature_len, channel=1, k=None, h=None):
        """
        feature_len: 每个样本的电压特征数量（例如 n_buses-1）
        channel: 输入通道数（通常为1）
        k: 漂移参数（drift parameter），若为 None，则在 fit 时设定为 0.5 * baseline_std
        h: 累计和阈值（threshold），若为 None，则在 fit 时设定为 5 * baseline_std
        """
        self.num_features = channel * feature_len
        self.k = k
        self.h = h

    def fit(self, x, y=None):
        """
        x: numpy 数组或 PyTorch 张量，形状为 (n_samples, channel, feature_len)
        计算正常状态下所有样本展平后的总体均值与标准差，并自适应设定 k 和 h（若未提供）
        """
        if isinstance(x, torch.Tensor):
            x = x.numpy()
        n_samples = x.shape[0]
        x_flat = x.reshape(n_samples, -1)
        self.baseline_mean = np.mean(x_flat)
        self.baseline_std = np.std(x_flat)
        if self.k is None:
            self.k = 0.5 * self.baseline_std
        if self.h is None:
            self.h = 5 * self.baseline_std

    def predict(self, x):
        """
        x: numpy 数组或 PyTorch 张量，形状为 (n_samples, channel, feature_len)
        对每个时间步（样本）计算累积偏差 s，
        s[t] = max(0, s[t-1] + (sample_mean[t] - baseline_mean - k))
        若 s[t] > h，则判定该时间步为 outage（异常），返回 1，否则返回 0
        """
        if isinstance(x, torch.Tensor):
            x = x.numpy()
        n_samples = x.shape[0]
        x_flat = x.reshape(n_samples, -1)
        sample_means = np.mean(x_flat, axis=1)
        s = np.zeros(n_samples)
        predictions = np.zeros(n_samples, dtype=int)
        s[0] = max(0, sample_means[0] - self.baseline_mean - self.k)
        if s[0] > self.h:
            predictions[0] = 1
        for t in range(1, n_samples):
            s[t] = max(0, s[t-1] + (sample_means[t] - self.baseline_mean - self.k))
            predictions[t] = 1 if s[t] > self.h else 0
        return predictions

    def predict_proba(self, x):
        """
        x: numpy 数组或 PyTorch 张量，形状为 (n_samples, channel, feature_len)
        返回: numpy 数组，形状为 (n_samples, 2)，每行分别表示正常 (0) 和 outage (1) 的概率
        利用累积偏差 s 与阈值 h 的差值经过 sigmoid 映射得到 outage 概率，
        当 s 明显低于 h 时 outage 概率低，反之则高。
        """
        if isinstance(x, torch.Tensor):
            x = x.numpy()
        n_samples = x.shape[0]
        x_flat = x.reshape(n_samples, -1)
        sample_means = np.mean(x_flat, axis=1)
        s = np.zeros(n_samples)
        s[0] = max(0, sample_means[0] - self.baseline_mean - self.k)
        for t in range(1, n_samples):
            s[t] = max(0, s[t-1] + (sample_means[t] - self.baseline_mean - self.k))
        diff = s - self.h
        p_outage = 1 / (1 + np.exp(-diff))
        p_normal = 1 - p_outage
        return np.column_stack((p_normal, p_outage))

In [3]:
dataset = BinaryDataset('Node123_loop')
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size], generator=torch.Generator().manual_seed(20))
feature_names = [f'Bus {i+1}' for i in range(1, dataset.n_buses + 1)]

In [4]:
model = VoltageCUSUMDetector(feature_len=dataset.n_buses)

In [5]:
model.fit(train_dataset.dataset.data, train_dataset.dataset.labels)

In [6]:
# Get predictions
y_true = test_dataset.dataset.labels[test_dataset.indices]
y_pred = model.predict(test_dataset.dataset.data[test_dataset.indices])
y_proba = model.predict_proba(test_dataset.dataset.data[test_dataset.indices])[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_proba)

# Print metrics
print(f'Accuracy: {accuracy}')
print(f'Recall: {recall}')
print(f'F1 Score: {f1}')
print(f'AUC: {auc}')

Accuracy: 0.5
Recall: 0.0
F1 Score: 0.0
AUC: 0.5
